## TODO: Translate for future english speaking students

In [ ]:
from pathlib import Path 
import sys

if "google.colab" in sys.modules:

    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/EnjeuxDecarbonationSante/")

else:
    DATA_DIR = Path(".")

In [ ]:
%%bash
cd /content/drive/MyDrive/
git clone https://github.com/NicolasPetiot/EnjeuxDecarbonationSante.git

In [ ]:
%%bash
pip install pyrosetta_installer

In [ ]:
from pyrosetta_installer import  install_pyrosetta
install_pyrosetta()

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix

from tqdm.notebook import tqdm

In [ ]:
# Code de lecture de fichier PDB
def load_pdb(path:Path) -> pd.DataFrame:
    """
    Parses a lines from a PDB file to extract DataFrame of atomic informations.
    """
    lines = path.read_text().splitlines()
    data  = [read_format(line) for line in lines if line.startswith(("ATOM", "HETATM"))]
    df = pd.DataFrame(data, columns = ["record_name", "name", "alt", "resn", "chain", "resi", "insertion", "x", "y", "z", "occupancy", "b", "segi", "e", "q"])
    return df

def read_format(line:str) -> tuple[str, str, str, str, int, str, float, float, float, float, float, float, str, str, str]:
    """
    Parses a line from a PDB file to extract atomic data.

    Args:
        line (str): A single line from the PDB file, formatted according
            to the PDB specification.

    Returns:
        tuple[str, str, str, str, int, str, float, float, float, float,
        float, float, str, str, str]: A tuple containing atom information,
        including:
            - Atom name
            - Alternate location indicator
            - Residue name
            - Chain identifier
            - Residue sequence number
            - Insertion code
            - X, Y, and Z coordinates
            - Occupancy and temperature factor
            - Segment identifier, element symbol, and charge
    """
    return (
        line[:6].strip(),
        #int(line[6:11].strip()),
        line[12:16].strip(),        # Atom name
        line[16:17].strip(),        # Alternate location indicator
        line[17:20].strip() ,       # Residue name
        line[21:22].strip(),        # Chain identifier
        int(line[22:26].strip()),   # Residue sequence number
        line[26:27].strip(),        # Insertion code
        float(line[30:38].strip()), # X coordinate
        float(line[38:46].strip()), # Y coordinate
        float(line[46:54].strip()), # Z coordinate
        float(line[54:60].strip()) if line[54:60].strip() != "" else 1.0, # Occupancy
        float(line[60:66].strip()) if line[60:66].strip() != "" else 0.0, # Temperature factor
        line[72:76].strip(),        # Segment identifier
        line[76:78].strip(),        # Element symbol
        line[78:80].strip()         # Charge on the atom
    )



In [ ]:
# Fonctions d'initialisation de Rosetta
def rosetta_setup(verbosity = False, allow_overwrite = True, H_optimization = True, extra_params:list[Path] = []):
    flags = [
        f"-mute false" if verbosity else "-mute all",
        f"-ex1",
        f"-ex2",
        f"-no_optH {str(not H_optimization)}",
        f"-flip_HNQ true",
        f"-ignore_ligand_chi true",
        f"-overwrite" if allow_overwrite else "",
        f"-restore_pre_talaris_2013_behavior true"
    ]
    flags = " ".join(flags)

    extra_flag = []
    for path in extra_params:
        if not path.exists():
            print(f"WARNING: ignoring file {str(path)} (does not exists or is not acessible)")

        else:
            extra_flag.append(str(path))
    if len(extra_flag) > 0:
        flags += " -extra_res_fa " + " ".join(extra_flag)

    try:
        from pyrosetta import init

    except ImportError:
        raise ValueError("La cellule d'installation de PyRosetta doit être exécutée.")

    finally:
        init(flags)

def load_rosetta_pose(path:Path):
    from pyrosetta import pose_from_file
    if path.exists() and path.is_file():
        return pose_from_file(str(path))

    elif not path.is_file():
        raise FileNotFoundError(f"'{str(path)}' est un répertoire, pas un fichier")

    else:
        raise FileNotFoundError(f"Le fichier {str(path)} n'existe pas ou n'est pas accessible.")

In [ ]:
# Fonction Rosetta qui calcule l'affinité protéine-ligand
from pyrosetta import Pose, ScoreFunction, Vector1
from pyrosetta import get_fa_scorefxn
from pyrosetta.rosetta import protocols

def binding_affinity(pose:Pose, partners = "AB_C", scorefxn:ScoreFunction = None) -> float:
    """
    Separates two partners and returns the difference of energy between bounded and separated states.
    """
    if scorefxn is None:
        scorefxn = get_fa_scorefxn()

    bind_score = scorefxn(pose)

    # Split partners:
    split_pose = pose.clone()
    jump = 2
    step_size = 100

    protocols.docking.setup_foldtree(pose, partners, Vector1([-1,-1,-1]))
    trans_mover = protocols.rigid.RigidBodyTransMover(split_pose,jump)
    trans_mover.step_size(step_size)
    trans_mover.apply(split_pose)

    split_score = scorefxn(split_pose)

    return bind_score - split_score

In [ ]:
# Définition du protocole de mutation:
from pyrosetta import Pose
from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects

MUTATE_XML_STRING = """
<ROSETTASCRIPTS>
	<SCOREFXNS>
	</SCOREFXNS>
	<RESIDUE_SELECTORS>
        <Index name="to_mutate_A" resnums="{resi}A"/>
        <Index name="to_mutate_B" resnums="{resi}B"/>
        <Or name="to_mutate" selectors="to_mutate_A,to_mutate_B" />
	</RESIDUE_SELECTORS>
	<MOVERS>
        <MutateResidue name="mutate" residue_selector="to_mutate" new_res="{new_res}" preserve_atom_coords="false" />
	</MOVERS>
	<PROTOCOLS>
        <Add mover_name="mutate" />
	</PROTOCOLS>
	<OUTPUT />
</ROSETTASCRIPTS>
"""

def mutate(pose:Pose, aa_num:int, new_aa:str) -> Pose:
    """
    Apply a mutation to both chains A&B of an input pose.

    The mutation site is identified using an input residue index `resi:int`

    The new residue is specified using the single letter residue code `new_res`
    """
    xml = XmlObjects.create_from_string(MUTATE_XML_STRING.format(resi=aa_num, new_res=new_aa))
    protocol = xml.get_mover("ParsedProtocol")

    mutant = pose.clone()
    protocol.apply(mutant)
    return mutant

In [ ]:
# Définition du protocole de selection des mutations 
import random
def select_random_mutation(pose:Pose, allowed_mutations:dict[int, list[str]], force_change = False):
    # Select mutation site:
    mutations_sites = list(allowed_mutations.keys())
    resi = random.choice(mutations_sites)

    old_aa = pose.sequence()[resi-1] # One-letter code
    old_aa = three_letter_code[old_aa]

    allowed_residues = allowed_mutations[resi].copy()
    if force_change:
        allowed_residues.remove(old_aa)

    new_aa = random.choice(allowed_residues)

    return old_aa, resi, new_aa

In [ ]:
# Définition du critère de métropolis:
from math import exp
import random

def metropolis(delta:float, temp:float) -> bool:
    """

    """
    if delta < 0:
        return True

    if temp == 0.0:
        return False

    acceptation_probability = exp(-delta/temp)
    return random.uniform(0, 1) < acceptation_probability

In [ ]:
PDB = DATA_DIR / "PDB" 
file = PDB / "GSTD1+GSH.pdb"

if not file.exists():
    raise FileNotFoundError(f"Le fichier '{file}' n'est pas accessible ou n'existe pas")

structure = load_pdb(file)
structure.head()

In [ ]:
protein = structure.query("record_name == 'ATOM' and e != 'H'")
ligand = structure.query("resn == 'GSH' and e != 'H'")
seuil_contact = 4.0 # Distance à partir de laquelle on considère que deux atomes sont en contact.

xyz = ["x", "y", "z"] # Distances calculées à partir des coordonnées x, y et z.
dists = distance_matrix(protein[xyz], ligand[xyz])
I, J = np.where(dists < seuil_contact)

print("Selection avec PyMol:")
resi = protein.iloc[I].resi.unique()
print("select Gsite, resi " + "+".join(resi.astype(str)))

print("")

print("Table d'acides aminés")
print(resi)

protein.iloc[I]

In [ ]:
# Energie de la structure GSTD1+GSH.pdb
rosetta_setup(
    extra_params=[DATA_DIR / ".rosettafiles/GSH.params"], 
    verbosity=False, 
    H_optimization=False
)

from pyrosetta import get_score_function
scorefxn = get_score_function()

PDB  = DATA_DIR / "PDB" 
file = PDB / "GSTD1+GSH.pdb"
pose = load_rosetta_pose(file)

G_bound = scorefxn(pose)
print(f"G_lié:    {G_bound:.3f} R.E.U.")

file = PDB / "test_GSTD1+GSH_split.pdb"
pose = load_rosetta_pose(file)

G_unbound = scorefxn(pose)
print(f"G_séparé: {G_unbound:.3f} R.E.U.")

deltaG = G_bound - G_unbound
print(f"dG:          {deltaG:.3f} R.E.U.")

In [ ]:
rosetta_setup(
    extra_params=[DATA_DIR / ".rosettafiles/GSH.params"], 
    verbosity=False, 
    H_optimization=True
)

PDB  = DATA_DIR / "PDB" 
file = PDB / "GSTD1+GSH.pdb"
pose = load_rosetta_pose(file)

dG = binding_affinity(pose)
print(f"dG: {dG:.3f} R.E.U.")

In [ ]:
rosetta_setup(
    extra_params=[DATA_DIR / ".rosettafiles/GSH.params"], 
    verbosity=False, 
    H_optimization=True
)

PDB  = DATA_DIR / "PDB" 
file = PDB / "GSTD1+GSH.pdb"
pose = load_rosetta_pose(file)

dG = binding_affinity(pose)
print(f"dG: {dG:.3f} R.E.U.")

# Choix de la mutation:
new_aa = "Met" # TODO: À modifier
aa_num = 1     # TODO: À modifier
save_mutant = False

mutant = mutate(pose, aa_num=aa_num, new_aa=new_aa.upper())
dG_mutant = binding_affinity(mutant)
print(f"dG: {dG_mutant:.3f} R.E.U.")

if save_mutant:
    mutant_filename = str(pdb).replace(".pdb", f"_{new_aa}{aa_num}.pdb")
    mutant.dump_pdb(mutant_filename)

print(f"Mutation ddG: {dG_mutant - dG:.3f} R.E.U.")

In [ ]:
# Définition de notre espace des séquences
ALLOWED_MUTATIONS = {
    10: ["S", "G", "A", "C"], # Exemple
}

three_letter_code ={'V':'VAL', 'I':'ILE', 'L':'LEU', 'E':'GLU', 'Q':'GLN',
    'D':'ASP', 'N':'ASN', 'H':'HIS', 'W':'TRP', 'F':'PHE', 'Y':'TYR',
    'R':'ARG', 'K':'LYS', 'S':'SER', 'T':'THR', 'M':'MET', 'A':'ALA',
    'G':'GLY', 'P':'PRO', 'C':'CYS'}

# Conversion code à une lettre -> code à trois lettres
ALLOWED_MUTATIONS = {resi: [three_letter_code[aa] for aa in aas] for resi, aas in ALLOWED_MUTATIONS.items()}
print(ALLOWED_MUTATIONS)

In [ ]:
rosetta_setup(
    extra_params=[DATA_DIR / ".rosettafiles/GSH.params"], 
    verbosity=False, 
    H_optimization=True
)

PDB  = DATA_DIR / "PDB" 
file = PDB / "GSTD1+GSH.pdb"
pose = load_rosetta_pose(file)

dG = binding_affinity(pose)
print(f"dG: {dG:.3f} R.E.U.")

# Choix de la mutation:
old, resi, new = select_random_mutation(pose, ALLOWED_MUTATIONS, force_change=True)
print(f"Mutation: {old[0]}{old[1:].lower()}{resi} -> {new[0]}{new[1:].lower()}")

mutant = mutate(pose, aa_num=aa_num, new_aa=new_aa.upper())
dG_mutant = binding_affinity(mutant)
print(f"dG: {dG_mutant:.3f} R.E.U.")

if save_mutant:
    mutant_filename = str(pdb).replace(".pdb", f"_{new_aa}{aa_num}.pdb")
    mutant.dump_pdb(mutant_filename)

print(f"Mutation ddG: {dG_mutant - dG:.3f} R.E.U.")

In [ ]:
N_ITER = 500
PDB_INIT = DATA_DIR / "PDB" / "GSTD1+GSH.pdb"
TEMP = 0.2

# Initialisation:
ref = load_rosetta_pose(PDB_INIT)
dG_ref = binding_affinity(ref)

old_aa = [pd.NA]
indices = [pd.NA]
new_aa = [pd.NA]

dGs = [dG_ref]
ddGs = [pd.NA]
accepted = [True]

min_dG = float("inf") # Utile pour sauvegarder le meilleur design
save_mutant = False

for _ in tqdm(range(N_ITER)):
    # 1- Selection aléatoire de la mutation:
    old, resi, new = select_random_mutation(ref, ALLOWED_MUTATIONS, force_change=True)
    mutant = mutate(ref, resi, new)

    # 2- Affinité et ddG:
    dG_mutant = binding_affinity(mutant)
    ddG = dG_mutant - dG_ref

    # 3- Critère de Métropolis:
    is_accepted = metropolis(ddG, temp=TEMP)
    if is_accepted:
        ref = mutant.clone()
        dG_ref = dG_mutant

    # 4- Sauvegarde des informations pertinentes:
    old_aa.append(old)
    indices.append(resi)
    new_aa.append(new)

    dGs.append(dG_mutant)
    ddGs.append(ddG)
    accepted.append(is_accepted)

    if dG_mutant < min_dG:
        min_dG = dG_mutant
        best_pose = mutant.clone()

df = pd.DataFrame({
    "old": old_aa,
    "resi": indices,
    "new": new_aa,

    "dG": dGs,
    "ddG": ddGs,
    "accepted": accepted
})

if save_mutant:
    mutant_filename = str(PDB_INIT).replace(".pdb", f"_best_design.pdb")